In [14]:
import os
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from fastembed import TextEmbedding
import re
from rank_bm25 import BM25Okapi

load_dotenv()

True

#### LLM

In [2]:
API_KEY=os.getenv("OPENROUTER_API_KEY")
BASE_URL="https://openrouter.ai/api/v1"
MODEL="stepfun/step-3.5-flash:free"

In [3]:
def llm_completion(prompt):
    client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
    )
    
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            },
        ],
    )
    
    return completion.choices[0].message.content

### Chunking

In [4]:
def chunk_by_section(text):
    """Split document by markdown headers (##)"""
    pattern = r"\n## "
    chunks = re.split(pattern, text)
    return [chunk.strip() for chunk in chunks if chunk.strip()]

with open('./report.md','r') as f:
    text = f.read()

useful_chunks = chunk_by_section(text)

### Adding Context to Chunk

In [5]:
def get_relevant_context(full_text, target_chunk, start_chunks=2, surrounding_chunks=2):
    """
    Instead of sending entire document, send:
    - First few chunks (summary/abstract)
    - Chunks immediately before target
    """
    chunks = chunk_by_section(full_text)
    target_idx = chunks.index(target_chunk) if target_chunk in chunks else -1

    selected = []

    # Add first chunks (usually have overview)
    selected.extend(chunks[:start_chunks])

    # Add chunks before target
    if target_idx > 0:
        start = max(0, target_idx - surrounding_chunks)
        selected.extend(chunks[start:target_idx])

    # Remove duplicates and join
    unique = []
    for c in selected:
        if c not in unique:
            unique.append(c)

    return "\n\n".join(unique)


def add_context_to_chunk(chunk, source_text, start_chunks=2, surrounding_chunks=2):
    """
    Adding context to a chunk using LLM
    """
    # For large docs, include only relevant context
    context_text = get_relevant_context(
        source_text, chunk, start_chunks, surrounding_chunks
    )

    prompt = f"""
    You are adding context to a chunk of text from a larger document.

    Full document context:
    {context_text[:2000]}  # Truncate if needed

    Chunk to contextualize:
    {chunk}

    Write 1-2 sentences explaining where this chunk fits in the larger document
    and any important relationships to other sections. Be concise.

    Context:
    """

    context = llm_completion(prompt)

    # Storing the contextualized chunk
    return f"[Context: {context}] {chunk}"


contextualized_chunks = []
for chunk in useful_chunks:
    contextualized = add_context_to_chunk(
        chunk=chunk, source_text=text, start_chunks=2, surrounding_chunks=2
    )
    contextualized_chunks.append(
        {
            "original": chunk,
            "contextualized": contextualized,
            "id": f"doc_{len(contextualized_chunks):03d}",
        }
    )

# Now using contextualized chunks for everything else
chunk_content = [c["contextualized"] for c in contextualized_chunks]

### Embeddings

In [6]:
embedding_model = TextEmbedding()


def generate_embedding(texts):
    return list(embedding_model.embed(texts))


chunk_embeddings = [generate_embedding(chunk) for chunk in chunk_content]

### Vector store

In [7]:
class VectorIndex:
    def __init__(self):
        self.vectors = []
        self.metadata = []

    def add_vector(self, embeddings, metadata):
        self.vectors.append(np.array(embeddings).flatten())
        self.metadata.append(metadata)

    def search(self, query_embedding, top_k=2):
        query_emb = np.array(query_embedding)

        # Calculate similarities
        similarities = []
        for i, vec in enumerate(self.vectors):
            vec = np.array(vec)
            similarity = np.dot(query_emb, vec) / (
                np.linalg.norm(query_emb) * np.linalg.norm(vec)
            )
            similarities.append((similarity, i))

        # Sort by similarity (highest first)
        similarities.sort(reverse=True)

        # Return top_k results
        results = []
        for sim, idx in similarities[:top_k]:
            results.append(
                {
                    "distance": 1 - sim,
                    "content": self.metadata[idx]["content"],
                    "similarity": sim,
                }
            )
        return results


store = VectorIndex()

# Now looping through each chunk and it's embeddings
for embedding, chunk in zip(chunk_embeddings, chunk_content):
    store.add_vector(embedding, {"content": chunk})

### User quey

In [8]:
user_qn = "What did the software engineering department do last year?"
question_embedding = list(embedding_model.embed([user_qn]))[0]

### BM25 index

In [9]:
tokenized_chunks = [chunk.lower().split() for chunk in chunk_content]

# BM25 index
bm25 = BM25Okapi(tokenized_chunks)

In [10]:
def hybrid_search(query, chunks, bm25, vector_store, top_k=3):
    # 1. Getting BM25 results
    tokenized_query = query.lower().split()
    bm25_results = bm25.get_top_n(tokenized_query, chunks, n=top_k * 2)

    # 2. Getting semantic results
    query_emb = list(embedding_model.embed([query]))[0]
    semantic_results = vector_store.search(query_emb, top_k=top_k * 2)
    # print(semantic_results)

    # 3. Combining and deduplicating
    combined = {}

    # Adding BM25 results with height
    for i, chunk in enumerate(bm25_results):
        combined[chunk] = combined.get(chunk, 0) + (top_k * 2 - i)

    # Adding semantic results with weight
    for i, result in enumerate(semantic_results):
        chunk = result['content']
        combined[chunk] = combined.get(chunk, 0) + (top_k * 2 - i)

    # Sorting by combines score and returning top_k
    sorted_chunks = sorted(combined.items(), key=lambda x: x[1], reverse=True)
    return [chunk for chunk, score in sorted_chunks[:top_k]]

In [11]:
top_chunks = hybrid_search(
    query="incident-2023 software engineering",
    chunks=chunk_content,
    bm25=bm25,
    vector_store=store,
)

top_chunks

["[Context: This chunk is the detailed Section 2 for Software Engineering, expanding on the Executive Summary's brief mention of stability fixes for Project Phoenix, specifically addressing error codes like `ERR_MEM_ALLOC_FAIL_0x8007000E`. It highlights critical dependencies with Product Engineering (Section 6) and underscores interdisciplinary collaboration by showing how software improvements support broader product development efforts.] Section 2: Software Engineering - Project Phoenix Stability Enhancements\n\nThe Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems underpinning Project Phoenix. Recurring issues, particularly `ERR_MEM_ALLOC_FAIL_0x8007000E` during peak loads and `TIMEOUT_QUERY_DB_0xDEADBEEF` affecting data retrieval operations, were prioritized, at a cost of INC-2023-Q4-011. Root cause analysis pointed towards inefficiencies in the primary data caching algorithm and suboptimal database indexing s

### LLM calling 🤙 .....

In [12]:
context = top_chunks[0]
print(context)

final_prompt = f"""You are a helpful assistant answering questions based on provided context.

CONTEXT:
{context}

USER QUESTION:
{user_qn}

Answer the question based ONLY on the context provided. If the answer cannot be found in the context, say "I cannot find this information in the provided documents."
"""

[Context: This chunk is the detailed Section 2 for Software Engineering, expanding on the Executive Summary's brief mention of stability fixes for Project Phoenix, specifically addressing error codes like `ERR_MEM_ALLOC_FAIL_0x8007000E`. It highlights critical dependencies with Product Engineering (Section 6) and underscores interdisciplinary collaboration by showing how software improvements support broader product development efforts.] Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems underpinning Project Phoenix. Recurring issues, particularly `ERR_MEM_ALLOC_FAIL_0x8007000E` during peak loads and `TIMEOUT_QUERY_DB_0xDEADBEEF` affecting data retrieval operations, were prioritized, at a cost of INC-2023-Q4-011. Root cause analysis pointed towards inefficiencies in the primary data caching algorithm and suboptimal database indexing strat

In [13]:
res = llm_completion(prompt=final_prompt)
res

"Based on the provided context, the software engineering department's activities last year (referencing Q4 2024) included:\n\n- Dedicating effort to improve stability and performance for Project Phoenix by prioritizing and addressing recurring errors like `ERR_MEM_ALLOC_FAIL_0x8007000E` and `TIMEOUT_QUERY_DB_0xDEADBEEF`.\n- Conducting root cause analysis that identified inefficiencies in the data caching algorithm and suboptimal database indexing strategies.\n- Deploying a patch for the memory allocation error, which led to a 40% reduction in critical failures under simulated stress tests in Q4 2024 (Test Case ID: INC-2023-Q4-011).\n- Scheduling refactoring of the query module for the next release cycle to resolve the timeout issue.\n- Emphasizing robust testing protocols due to dependencies with Product Engineering and continuously monitoring system telemetry for regressions.\n- Assisting with the INC-2023-Q4-011 incident during Q4 2024.\n\nThese actions were part of the stability enh